# 1. Data Preparation

This notebook builds the dataset used in every experiment in the project:

* **50,000 database sentences** and **500 query sentences** sampled from English Wikipedia
  ([`sentence-transformers/wikipedia-en-sentences`](https://huggingface.co/datasets/sentence-transformers/wikipedia-en-sentences)).
* A **384-dimensional embedding** of every sentence, computed with the `all-MiniLM-L6-v2` sentence-transformer model.

Nearest-neighbour search on these vectors finds sentences with similar *meaning*, which makes it a realistic high-dimensional search problem.

**Output files** (in `wikipedia_nn_data/`):

| File | Contents |
|---|---|
| `sentences.json` | the 50,000 database sentences |
| `query_sentences.json` | the 500 query sentences |
| `embeddings.npy` | database embeddings, shape `(50000, 384)`, `float32` |
| `queries.npy` | query embeddings, shape `(500, 384)`, `float32` |

> **Note:** these files are already included in the repository, so this notebook does not need to be re-run to reproduce the other notebooks. Re-running it downloads data from Hugging Face and encodes 50,500 sentences, which takes several minutes.

## Setup

Install the required packages. The second and third cells fixed network/decompression errors in our environment when streaming from Hugging Face; they may not be needed elsewhere.

In [ ]:
# One-time setup: install the packages used in this project.
# Can be skipped if they are already installed.
%pip install datasets sentence-transformers numpy scipy matplotlib pandas

In [ ]:
# Environment fix: upgrade the HTTP libraries used by `datasets` for streaming.
%pip install --upgrade httpcore httpx h11

In [ ]:
# Environment fix: Brotli is needed to decompress some Hugging Face responses.
%pip install --upgrade brotli

## Load the dataset

The dataset is opened in **streaming** mode, so rows are downloaded on demand instead of fetching the whole corpus.

In [ ]:
# Stream the dataset instead of downloading it in full: the corpus has millions
# of sentences and we only need ~50k, so rows are fetched lazily as we iterate.
from datasets import load_dataset

dataset = load_dataset(
    "sentence-transformers/wikipedia-en-sentences",
    split="train",
    streaming=True
)

In [3]:
# Quick look at the raw data.
for row in dataset.take(5):
    print(row["sentence"])

The film stars M. G. Ramachandran, Latha, Anjali Devi and V. S. Raghavan in lead roles.
Naarda plenirena is a species of moth in the family Noctuidae.
Sponsored by the American Federation of Labor and Congress of Industrial Organizations (AFL-CIO), labor groups paid the entire $40,000 cost of the memorial.
Since that election the Belfast Corporation Act 1896 had increased the borough from 6,800 to 16,500 acres, and the electorate from 39,603 to 47,294.
It was also included on their Best of Volume 1.


## Sample the sentences

We shuffle the stream with a fixed seed and keep sentences until we have 50,500. Two filters are applied:

* **At least 5 words**: very short fragments (headings, single names) carry little meaning and make poor search examples.
* **No duplicates**: duplicates would produce identical vectors and could place a query inside the database.

In [ ]:
from tqdm.auto import tqdm

N = 50_000          # database size
N_QUERIES = 500     # number of held-out query sentences
SEED = 42           # fixed seed for reproducibility

# Streaming shuffle: rows are shuffled inside a 100k-row buffer. This is an
# approximate shuffle, since the full dataset cannot be held in memory.
shuffled_dataset = dataset.shuffle(
    seed=SEED,
    buffer_size=100_000
)

sentences = []
seen = set()

with tqdm(total=N + N_QUERIES) as progress:
    for row in shuffled_dataset:
        text = row["sentence"].strip()

        # Keep sentences with at least 5 words and skip exact duplicates
        if len(text.split()) >= 5 and text not in seen:
            sentences.append(text)
            seen.add(text)
            progress.update(1)

        # Stop once we have enough sentences for the database and the queries
        if len(sentences) == N + N_QUERIES:
            break

print("Total sentences:", len(sentences))

## Split into database and queries

The database is what we search *in*; the queries are what we search *for*. Keeping them disjoint means each query is a new, unseen sentence, which is how a search system is used in practice.

In [5]:
# The first N sentences form the database, the remaining N_QUERIES are queries.
# Duplicates were removed above, so no query also appears in the database.
database_sentences = sentences[:N]
query_sentences = sentences[N:]

print("Database:", len(database_sentences))
print("Queries:", len(query_sentences))

Database: 50000
Queries: 500


In [6]:
# Save the raw text so later notebooks can show the sentences behind search results.
import json
from pathlib import Path

DATA_DIR = Path("wikipedia_nn_data")
DATA_DIR.mkdir(exist_ok=True)

with open(DATA_DIR / "sentences.json", "w") as f:
    json.dump(database_sentences, f)

with open(DATA_DIR / "query_sentences.json", "w") as f:
    json.dump(query_sentences, f)

## Compute embeddings

`all-MiniLM-L6-v2` maps each sentence to a 384-dimensional vector so that sentences with similar meaning end up close together.

We **normalise every embedding to unit length**. For unit vectors,

$$\|a - b\|^2 = 2 - 2\cos(a, b),$$

so ranking neighbours by Euclidean distance gives exactly the same result as ranking by cosine similarity. This lets us use standard Euclidean search structures such as the KD-tree.

In [ ]:
# all-MiniLM-L6-v2: a small, fast sentence-embedding model producing 384-dim vectors.
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [8]:
# Embed a single sentence to inspect the output format.
sentence = database_sentences[0]

embedding = model.encode(
    sentence,
    normalize_embeddings=True
)

print("Sentence:", sentence)
print("Embedding shape:", embedding.shape)
print("First 10 coordinates:", embedding[:10])

Sentence: Gray Hill Esq and his wife, also traveling in 1895, met the Hameideh south of Madeba.
Embedding shape: (384,)
First 10 coordinates: [-0.01246414  0.10611856 -0.04010179  0.03397487 -0.08948258  0.02738277
 -0.07015882  0.05279549 -0.04784937 -0.02131057]


Encode the full database and the queries in batches of 64.

In [ ]:
# Embed the whole database. normalize_embeddings=True scales every vector to unit
# length, so Euclidean distance ranks neighbours the same way as cosine similarity.
X = model.encode(
    database_sentences,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Database shape:", X.shape)

In [ ]:
# Embed the queries with exactly the same settings.
Q = model.encode(
    query_sentences,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Query shape:", Q.shape)

## Check and save

Confirm the vectors are `float32` and have unit length, then save them for the following notebooks.

In [12]:
# Sanity checks: data type and unit length of every vector.
print("Database dtype:", X.dtype)

print("Database normalized:",
      np.allclose(np.linalg.norm(X, axis=1), 1))

print("Queries normalized:",
      np.allclose(np.linalg.norm(Q, axis=1), 1))

Database dtype: float32
Database normalized: True
Queries normalized: True


In [13]:
# Save as .npy files so later notebooks can load the embeddings without re-encoding.
np.save(DATA_DIR / "embeddings.npy", X)
np.save(DATA_DIR / "queries.npy", Q)

print("Embeddings saved successfully.")

Embeddings saved successfully.


## Summary

* 50,000 database and 500 query sentences, all distinct and at least 5 words long.
* Every sentence is a unit-length 384-dimensional `float32` vector.
* Everything is saved to `wikipedia_nn_data/`. The next notebook (`02_brute_force_baseline.ipynb`) computes the exact nearest neighbours.